# **Conceito**

Um sistema de recomendação baseado em conteúdo (Content-Based) recomenda itens ao usuário com base na similaridade entre o perfil do usuário e as características dos itens. O KNN entra como o motor de busca: para um dado usuário, encontramos os K itens mais similares ao seu perfil.

## **Recomendação de Filmes**
Vamos construir um sistema que recomenda filmes a um usuário com base em:

* **Features dos filmes:** gênero (ação, comédia, drama, etc.), duração, ano, diretor.

*  **Perfil do usuário**: filmes que ele já assistiu e avaliou.

## **Catálogo de filmes**

In [ ]:
import pandas as pd
import numpy as np

filmes = pd.DataFrame({
    'filme_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'titulo': ['Matrix', 'Toy Story', 'Titanic', 'John Wick', 'Shrek',
               'Interestelar', 'O Poderoso Chefão', 'Se Beber Não Case',
               'Vingadores', 'La La Land'],
    'acao':     [1, 0, 0, 1, 0, 1, 0, 0, 1, 0],
    'comedia':  [0, 1, 0, 0, 1, 0, 0, 1, 0, 0],
    'drama':    [1, 0, 1, 0, 0, 1, 1, 0, 0, 1],
    'romance':  [0, 0, 1, 0, 0, 0, 0, 0, 0, 1],
    'ficcao':   [1, 0, 0, 0, 0, 1, 0, 0, 1, 0],
    'animacao': [0, 1, 0, 0, 1, 0, 0, 0, 0, 0],
    'ano':      [1999, 1995, 1997, 2014, 2001, 2014, 1972, 2009, 2012, 2016]
})



In [ ]:
print(filmes)

   filme_id             titulo  acao  comedia  drama  romance  ficcao  \
0         1             Matrix     1        0      1        0       1   
1         2          Toy Story     0        1      0        0       0   
2         3            Titanic     0        0      1        1       0   
3         4          John Wick     1        0      0        0       0   
4         5              Shrek     0        1      0        0       0   
5         6       Interestelar     1        0      1        0       1   
6         7  O Poderoso Chefão     0        0      1        0       0   
7         8  Se Beber Não Case     0        1      0        0       0   
8         9         Vingadores     1        0      0        0       1   
9        10         La La Land     0        0      1        1       0   

   animacao   ano  
0         0  1999  
1         1  1995  
2         0  1997  
3         0  2014  
4         1  2001  
5         0  2014  
6         0  1972  
7         0  2009  
8         0  201

## **O que o usuário assistiu e  avaliou?**

In [ ]:
historico = pd.DataFrame({
    'usuario': ['U001', 'U001', 'U001', 'U001'],
    'filme_id': [1, 6, 9, 4],       # Matrix, Interestelar, Vingadores, John Wick
    'nota': [5, 5, 4, 4]
})
print(historico)

  usuario  filme_id  nota
0    U001         1     5
1    U001         6     5
2    U001         9     4
3    U001         4     4


## **Construção do Perfil do Usuário**

O perfil é a média ponderada das features dos filmes assistidos, usando as notas como pesos.

In [ ]:
# Features dos filmes (sem id, titulo, ano)
features_cols = ['acao', 'comedia', 'drama', 'romance', 'ficcao', 'animacao']

In [ ]:
# Merge do histórico com features dos filmes
hist_features = historico.merge(filmes, on='filme_id')

In [ ]:
hist_features

,usuario,filme_id,nota,titulo,acao,comedia,drama,romance,ficcao,animacao,ano
0,U001,1,5,Matrix,1,0,1,0,1,0,1999
1,U001,6,5,Interestelar,1,0,1,0,1,0,2014
2,U001,9,4,Vingadores,1,0,0,0,1,0,2012
3,U001,4,4,John Wick,1,0,0,0,0,0,2014


In [ ]:
# Perfil = média ponderada pelas notas
perfil = {}
for col in features_cols:
    perfil[col] = np.average(hist_features[col], weights=hist_features['nota'])

perfil = pd.Series(perfil)


In [ ]:
print("Perfil do usuário U001:")
print(perfil.round(3))

Perfil do usuário U001:
acao        1.000
comedia     0.000
drama       0.556
romance     0.000
ficcao      0.778
animacao    0.000
dtype: float64


## **Matriz de features dos filmes**

In [ ]:
from sklearn.preprocessing import StandardScaler,MinMaxScaler

# Features dos filmes (sem ano para simplificar)
X_filmes = filmes[features_cols].values

In [ ]:
X_filmes

array([[1, 0, 1, 0, 1, 0],
       [0, 1, 0, 0, 0, 1],
       [0, 0, 1, 1, 0, 0],
       [1, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 1],
       [1, 0, 1, 0, 1, 0],
       [0, 0, 1, 0, 0, 0],
       [0, 1, 0, 0, 0, 0],
       [1, 0, 0, 0, 1, 0],
       [0, 0, 1, 1, 0, 0]])

In [ ]:
# Padronizar
scaler = MinMaxScaler()
X_filmes_scaled = scaler.fit_transform(X_filmes)

In [ ]:
# Perfil do usuário (mesmo espaço)
perfil_scaled = scaler.transform(perfil.values.reshape(1, -1))

##  **KNN com similaridade de cosseno**

In [ ]:
from sklearn.neighbors import NearestNeighbors

# KNN usando distância de cosseno
knn = NearestNeighbors(
    n_neighbors=7,           # buscar 7 vizinhos mais próximos
    metric='cosine',         # distância de cosseno
    algorithm='brute'
    )

##  **Ajustar o modelo**

In [ ]:

knn.fit(X_filmes_scaled)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=7)

## **Buscar os K filmes mais similares ao perfil**

In [ ]:
# Buscar os K filmes mais similares ao perfil
distancias, indices = knn.kneighbors(perfil_scaled)

In [ ]:
distancias, indices

(array([[0.02614832, 0.02614832, 0.09126107, 0.2771026 , 0.59839034,
         0.71601908, 0.71601908]]),
 array([[0, 5, 8, 3, 6, 2, 9]]))

In [ ]:
# Converter distância em similaridade (1 - distância)
similaridades = 1 - distancias[0]/2


## **Calculo da similaridade**

In [ ]:
similaridades

array([0.98692584, 0.98692584, 0.95436947, 0.8614487 , 0.70080483,
       0.64199046, 0.64199046])

## **Resultado**

In [ ]:

# Resultado
recomendacoes = filmes.iloc[indices[0]].copy()
recomendacoes['similaridade'] = similaridades
recomendacoes = recomendacoes.sort_values('similaridade', ascending=False)

print(recomendacoes[['titulo', 'similaridade']])

              titulo  similaridade
0             Matrix      0.986926
5       Interestelar      0.986926
8         Vingadores      0.954369
3          John Wick      0.861449
6  O Poderoso Chefão      0.700805
2            Titanic      0.641990
9         La La Land      0.641990


##  **Remover os filmes que já foram assitidos**

In [ ]:
#
ja_vistos = historico['filme_id'].tolist()
recomendacoes_finais = recomendacoes[~recomendacoes['filme_id'].isin(ja_vistos)]

print("Recomendações para U001:")
print(recomendacoes_finais[['titulo', 'similaridade']].head(3))

🎬 Recomendações para U001:
              titulo  similaridade
6  O Poderoso Chefão      0.700805
2            Titanic      0.641990
9         La La Land      0.641990
